In [4]:
import pandas as pd
import numpy as np

historical_results = pd.read_csv('../data/processed/historical_results.csv')

# FIFA rankings
fifa_rankings_clean = pd.read_csv('../data/processed/fifa_rankings_clean.csv')

# Transfermarkt files
players_clean = pd.read_csv('../data/processed/players_clean.csv')
player_valuations_clean = pd.read_csv(
        '../data/processed/player_valuations_clean.csv')
national_teams_clean = pd.read_csv('../data/processed/national_teams_clean.csv')
games = pd.read_csv('../data/raw/transfermarkt/games.csv')
game_lineups = pd.read_csv(
        '../data/raw/transfermarkt/game_lineups.csv', low_memory=False)
    

In [2]:
print(historical_results['date'].dtype)
print(fifa_rankings_clean['rank_date'].dtype)

str
str


In [3]:
historical_results['date'] = pd.to_datetime(historical_results['date'])
fifa_rankings_clean['rank_date'] = pd.to_datetime(fifa_rankings_clean['rank_date'])

In [4]:
print(historical_results['date'].dtype)
print(fifa_rankings_clean['rank_date'].dtype)

datetime64[us]
datetime64[us]


In [8]:
historical_results = historical_results.sort_values(by='date')
fifa_rankings_clean = fifa_rankings_clean.sort_values(by='rank_date')

home_trimmed_ranking = fifa_rankings_clean[['country_full', 'rank_date', 'rank', 'total_points']].copy()
home_trimmed_ranking = home_trimmed_ranking.rename(columns = {'country_full' : 'home_team', 'rank' : 'home_rank', 'total_points' : 'home_points'})
historical_results = pd.merge_asof(historical_results, home_trimmed_ranking, left_on = 'date', right_on = 'rank_date', by = 'home_team', direction = 'backward')

away_trimmed_ranking = fifa_rankings_clean[['country_full', 'rank_date', 'rank', 'total_points']].copy()
away_trimmed_ranking = away_trimmed_ranking.rename(columns = {'country_full' : 'away_team', 'rank' : 'away_rank', 'total_points' : 'away_points'})
historical_results = pd.merge_asof(historical_results, away_trimmed_ranking, left_on = 'date', right_on = 'rank_date', by = 'away_team', direction = 'backward')

In [7]:
historical_results[historical_results['date'] > '1992-12-31'][['date', 'home_team', 'away_team', 'rank', 'total_points']].head(10)

,date,home_team,away_team,rank,total_points
18708,1993-01-01,Ghana,Mali,39.0,34.0
18709,1993-01-02,Gabon,Burkina Faso,55.0,27.0
18710,1993-01-02,Kuwait,Lebanon,71.0,21.0
18711,1993-01-03,Burkina Faso,Mali,97.0,11.0
18712,1993-01-03,Gabon,Ghana,55.0,27.0
18713,1993-01-08,Uganda,Tanzania,92.0,12.0
18714,1993-01-10,Uganda,Tanzania,92.0,12.0
18715,1993-01-10,Tunisia,Bulgaria,38.0,35.0
18716,1993-01-10,Senegal,Algeria,51.0,27.0
18717,1993-01-10,Angola,Zimbabwe,102.0,10.0


In [9]:
historical_results[historical_results['date'] > '1992-12-31'][['date', 'home_team', 'away_team', 'home_rank', 'away_rank', 'home_points', 'away_points']].head(10)

,date,home_team,away_team,home_rank,away_rank,home_points,away_points
18708,1993-01-01,Ghana,Mali,39.0,69.0,34.0,22.0
18709,1993-01-02,Gabon,Burkina Faso,55.0,97.0,27.0,11.0
18710,1993-01-02,Kuwait,Lebanon,71.0,161.0,21.0,0.0
18711,1993-01-03,Burkina Faso,Mali,97.0,69.0,11.0,22.0
18712,1993-01-03,Gabon,Ghana,55.0,39.0,27.0,34.0
18713,1993-01-08,Uganda,Tanzania,92.0,80.0,12.0,15.0
18714,1993-01-10,Congo DR,Cameroon,NaN,22.0,NaN,43.0
18715,1993-01-10,Botswana,South Africa,139.0,124.0,2.0,5.0
18716,1993-01-10,Angola,Zimbabwe,102.0,54.0,10.0,27.0
18717,1993-01-10,Uganda,Tanzania,92.0,80.0,12.0,15.0


In [10]:
historical_results['points_diff'] = historical_results['home_points'] - historical_results['away_points']

In [11]:
historical_results[historical_results['date'] > '1992-12-31'][['home_team', 'away_team', 'home_points', 'away_points', 'points_diff']].head(10)

,home_team,away_team,home_points,away_points,points_diff
18708,Ghana,Mali,34.0,22.0,12.0
18709,Gabon,Burkina Faso,27.0,11.0,16.0
18710,Kuwait,Lebanon,21.0,0.0,21.0
18711,Burkina Faso,Mali,11.0,22.0,-11.0
18712,Gabon,Ghana,27.0,34.0,-7.0
18713,Uganda,Tanzania,12.0,15.0,-3.0
18714,Congo DR,Cameroon,NaN,43.0,NaN
18715,Botswana,South Africa,2.0,5.0,-3.0
18716,Angola,Zimbabwe,10.0,27.0,-17.0
18717,Uganda,Tanzania,12.0,15.0,-3.0


In [12]:
import numpy as np

home_form = historical_results[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'outcome']].copy()
home_form = home_form.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_score': 'goals_scored',
    'away_score': 'goals_conceded'
})

conditions = [
    home_form['outcome'] == 'Home Win',
    home_form['outcome'] == 'Away Win',
    home_form['outcome'] == 'Draw'
]

choices = [3, 0, 1]

home_form['form_points'] = np.select(conditions, choices, default=0)

In [13]:
home_form.head()

,date,team,opponent,goals_scored,goals_conceded,outcome,form_points
0,1872-11-30,Scotland,England,0.0,0.0,Draw,1
1,1873-03-08,England,Scotland,4.0,2.0,Home Win,3
2,1874-03-07,Scotland,England,2.0,1.0,Home Win,3
3,1875-03-06,England,Scotland,2.0,2.0,Draw,1
4,1876-03-04,Scotland,England,3.0,0.0,Home Win,3


In [14]:
away_form = historical_results[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'outcome']].copy()
away_form = away_form.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_score': 'goals_scored',
    'home_score': 'goals_conceded'
})

conditions = [
    away_form['outcome'] == 'Away Win',
    away_form['outcome'] == 'Home Win',
    away_form['outcome'] == 'Draw'
]

choices = [3, 0, 1]

away_form['form_points'] = np.select(conditions, choices, default=0)

In [15]:
away_form.head()

,date,opponent,team,goals_conceded,goals_scored,outcome,form_points
0,1872-11-30,Scotland,England,0.0,0.0,Draw,1
1,1873-03-08,England,Scotland,4.0,2.0,Home Win,0
2,1874-03-07,Scotland,England,2.0,1.0,Home Win,0
3,1875-03-06,England,Scotland,2.0,2.0,Draw,1
4,1876-03-04,Scotland,England,3.0,0.0,Home Win,0


In [16]:
team_form = pd.concat([home_form, away_form], ignore_index = True)

In [17]:
team_form.shape

(98678, 7)

In [19]:
team_form = team_form.sort_values(by=['team', 'date'])

team_form['form_5'] = (team_form
    .groupby('team')['form_points']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean()))

team_form['form_10'] = (team_form
    .groupby('team')['form_points']
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean()))

In [20]:
# Check a single team's form over time
team_form[team_form['team'] == 'Brazil'][['date', 'opponent', 'form_points', 'form_5', 'form_10']].head(15)

,date,opponent,form_points,form_5,form_10
49788,1914-09-20,Argentina,0,0.000000,0.000000
49789,1914-09-27,Argentina,3,1.500000,1.500000
482,1916-07-08,Chile,1,1.333333,1.333333
49822,1916-07-10,Argentina,1,1.250000,1.250000
485,1916-07-12,Uruguay,0,1.000000,1.000000
49827,1916-07-18,Uruguay,3,1.600000,1.333333
49855,1917-10-03,Argentina,0,1.000000,1.142857
49859,1917-10-07,Uruguay,0,0.800000,1.000000
521,1917-10-12,Chile,3,1.200000,1.222222
49863,1917-10-16,Uruguay,0,1.200000,1.100000


In [21]:
form_features = team_form[['date', 'team', 'form_5', 'form_10']].copy()

In [ ]:
historical_results = pd.merge(historical_results, form_features, left_on=['date', 'home_team'], right_on=['date', 'team'])
historical_results = historical_results.rename(columns={
    'form_5': 'home_form_5',
    'form_10': 'home_form_10'
})

historical_results = pd.merge(historical_results, form_features, left_on=['date', 'away_team'], right_on=['date', 'team'])
historical_results = historical_results.rename(columns={
    'form_5': 'away_form_5',
    'form_10': 'away_form_10'
})


In [23]:
historical_results[['date', 'home_team', 'away_team', 'home_form_5', 'home_form_10', 'away_form_5', 'away_form_10']].head(10)

,date,home_team,away_team,home_form_5,home_form_10,away_form_5,away_form_10
0,1872-11-30,Scotland,England,1.000000,1.000000,1.000000,1.000000
1,1873-03-08,England,Scotland,2.000000,2.000000,0.500000,0.500000
2,1874-03-07,Scotland,England,1.333333,1.333333,1.333333,1.333333
3,1875-03-06,England,Scotland,1.250000,1.250000,1.250000,1.250000
4,1876-03-04,Scotland,England,1.600000,1.600000,1.000000,1.000000
5,1876-03-25,Scotland,Wales,2.000000,1.833333,0.000000,0.000000
6,1877-03-03,England,Scotland,0.800000,0.833333,2.600000,2.000000
7,1877-03-05,Wales,Scotland,0.000000,0.000000,2.600000,2.125000
8,1878-03-02,Scotland,England,3.000000,2.222222,0.200000,0.714286
9,1878-03-23,Scotland,Wales,3.000000,2.300000,0.000000,0.000000


In [24]:
historical_results.shape

(49675, 27)

In [25]:
duplicates = historical_results[historical_results.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)]
print(duplicates.shape)
duplicates[['date', 'home_team', 'away_team']].head(10)

(571, 27)


,date,home_team,away_team
69,1890-03-15,Northern Ireland,England
70,1890-03-15,Northern Ireland,England
71,1890-03-15,Wales,England
72,1890-03-15,Wales,England
77,1891-03-07,England,Wales
78,1891-03-07,England,Wales
79,1891-03-07,England,Northern Ireland
80,1891-03-07,England,Northern Ireland
85,1892-03-05,Northern Ireland,England
86,1892-03-05,Northern Ireland,England


In [28]:
historical_results = historical_results.drop_duplicates(subset=['date', 'home_team', 'away_team'], keep='first')

In [30]:
historical_results.shape

(49338, 30)

In [31]:
historical_results['team1'] = historical_results[['home_team', 'away_team']].min(axis=1)
historical_results['team2'] = historical_results[['home_team', 'away_team']].max(axis=1)

In [32]:
historical_results[['home_team', 'away_team', 'team1', 'team2']].head(10)

,home_team,away_team,team1,team2
0,Scotland,England,England,Scotland
1,England,Scotland,England,Scotland
2,Scotland,England,England,Scotland
3,England,Scotland,England,Scotland
4,Scotland,England,England,Scotland
5,Scotland,Wales,Scotland,Wales
6,England,Scotland,England,Scotland
7,Wales,Scotland,Scotland,Wales
8,Scotland,England,England,Scotland
9,Scotland,Wales,Scotland,Wales


In [35]:
# Sort by team pair and date
h2h = historical_results[['date', 'team1', 'team2', 'outcome']].sort_values('date').reset_index(drop=True).copy()

hist_sorted = historical_results.sort_values('date').reset_index(drop=True)

h2h['team1_win'] = (
    ((h2h['outcome'] == 'Home Win') & (hist_sorted['home_team'] == h2h['team1'])) |
    ((h2h['outcome'] == 'Away Win') & (hist_sorted['away_team'] == h2h['team1']))
).astype(int)

h2h['team2_win'] = (
    ((h2h['outcome'] == 'Home Win') & (hist_sorted['home_team'] == h2h['team2'])) |
    ((h2h['outcome'] == 'Away Win') & (hist_sorted['away_team'] == h2h['team2']))
).astype(int)

h2h['draw'] = (h2h['outcome'] == 'Draw').astype(int)

In [36]:
h2h[['date', 'team1', 'team2', 'outcome', 'team1_win', 'team2_win', 'draw']].head(10)

,date,team1,team2,outcome,team1_win,team2_win,draw
0,1872-11-30,England,Scotland,Draw,0,0,1
1,1873-03-08,England,Scotland,Home Win,1,0,0
2,1874-03-07,England,Scotland,Home Win,0,1,0
3,1875-03-06,England,Scotland,Draw,0,0,1
4,1876-03-04,England,Scotland,Home Win,0,1,0
5,1876-03-25,Scotland,Wales,Home Win,1,0,0
6,1877-03-03,England,Scotland,Away Win,0,1,0
7,1877-03-05,Scotland,Wales,Away Win,1,0,0
8,1878-03-02,England,Scotland,Home Win,0,1,0
9,1878-03-23,Scotland,Wales,Home Win,1,0,0


In [37]:
h2h['h2h_team1_wins'] = (h2h.groupby(['team1', 'team2'])['team1_win']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0)))

h2h['h2h_team2_wins'] = (h2h.groupby(['team1', 'team2'])['team2_win']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0)))

h2h['h2h_draw'] = (h2h.groupby(['team1', 'team2'])['draw']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0)))

In [38]:
h2h[h2h['team1'] == 'England'][h2h['team2'] == 'Scotland'][['date', 'outcome', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw']].head(15)

/var/folders/57/1yhv43hx6tl7nz22346tzzs80000gn/T/ipykernel_25393/3270284631.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  h2h[h2h['team1'] == 'England'][h2h['team2'] == 'Scotland'][['date', 'outcome', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw']].head(15)


,date,outcome,h2h_team1_wins,h2h_team2_wins,h2h_draw
0,1872-11-30,Draw,0.0,0.0,0.0
1,1873-03-08,Home Win,0.0,0.0,1.0
2,1874-03-07,Home Win,1.0,0.0,1.0
3,1875-03-06,Draw,1.0,1.0,1.0
4,1876-03-04,Home Win,1.0,1.0,2.0
6,1877-03-03,Away Win,1.0,2.0,2.0
8,1878-03-02,Home Win,1.0,3.0,2.0
11,1879-04-05,Home Win,1.0,4.0,2.0
13,1880-03-13,Home Win,2.0,4.0,2.0
17,1881-03-12,Away Win,2.0,5.0,2.0


In [39]:
h2h_features = h2h[['date', 'team1', 'team2', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw']].copy()

In [ ]:
historical_results.head

In [42]:
historical_results = pd.merge(historical_results, h2h_features, on=['date', 'team1', 'team2'])

In [45]:
historical_results = historical_results.drop_duplicates(subset=['date', 'home_team', 'away_team'], keep='first')

In [46]:
historical_results.shape

(49338, 35)

In [51]:
historical_results.columns.tolist()

['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral',
 'outcome',
 'tournament_weight',
 'home_rank',
 'home_points',
 'away_rank',
 'away_points',
 'points_diff',
 'home_form_5',
 'home_form_10',
 'away_form_5',
 'away_form_10',
 'team1',
 'team2',
 'h2h_team1_wins',
 'h2h_team2_wins',
 'h2h_draw']

In [48]:
historical_results = historical_results.drop(columns=[
    'rank_date_x', 'rank_date_y', 'rank_date',
    'rank', 'total_points',
    'team_x', 'team_y'
])

In [50]:
historical_results = historical_results.drop(columns=['team'])
historical_results = historical_results.loc[:, ~historical_results.columns.duplicated()]

In [53]:
print(historical_results['home_rank'].isnull().sum())
print(historical_results['away_rank'].isnull().sum())
print(historical_results['home_points'].isnull().sum())
print(historical_results['away_points'].isnull().sum())

20939
21115
20939
21115


In [52]:
historical_results.to_csv('../data/processed/historical_results_features.csv', index=False)

In [55]:
historical_results = historical_results[historical_results['date'] > '1992-12-31']

In [56]:
historical_results.shape

(30631, 25)

In [ ]:
historical_results['home_rank'] = historical_results['home_rank'].astype(int)
historical_results['away_rank'] = historical_results['away_rank'].astype(int)

In [58]:
print(historical_results['home_rank'].isnull().sum())
print(historical_results['away_rank'].isnull().sum())

2232
2408


In [59]:
historical_results = historical_results.dropna(subset=['home_rank', 'away_rank'])

In [60]:
historical_results.shape

(27110, 25)

In [61]:
historical_results['home_rank'] = historical_results['home_rank'].astype(int)
historical_results['away_rank'] = historical_results['away_rank'].astype(int)

In [62]:
historical_results[['home_rank', 'away_rank', 'home_points', 'away_points']].head()

,home_rank,away_rank,home_points,away_points
18737,39,69,34.0,22.0
18738,55,97,27.0,11.0
18739,71,161,21.0,0.0
18740,97,69,11.0,22.0
18741,55,39,27.0,34.0


In [64]:
team_form['goal_diff'] = team_form['goals_scored'] - team_form['goals_conceded']

In [65]:
team_form = team_form.sort_values(by=['team', 'date'])

team_form['goal_diff_5'] = (team_form
    .groupby('team')['goal_diff']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean()))

team_form['goal_diff_10'] = (team_form
    .groupby('team')['goal_diff']
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean()))

In [66]:
team_form[team_form['team'] == 'Brazil'][['date', 'opponent', 'goals_scored', 'goals_conceded', 'goal_diff', 'goal_diff_5', 'goal_diff_10']].head(10)

,date,opponent,goals_scored,goals_conceded,goal_diff,goal_diff_5,goal_diff_10
49788,1914-09-20,Argentina,0.0,3.0,-3.0,-3.000000,-3.000000
49789,1914-09-27,Argentina,1.0,0.0,1.0,-1.000000,-1.000000
482,1916-07-08,Chile,1.0,1.0,0.0,-0.666667,-0.666667
49822,1916-07-10,Argentina,1.0,1.0,0.0,-0.500000,-0.500000
485,1916-07-12,Uruguay,1.0,2.0,-1.0,-0.600000,-0.600000
49827,1916-07-18,Uruguay,1.0,0.0,1.0,0.200000,-0.333333
49855,1917-10-03,Argentina,2.0,4.0,-2.0,-0.400000,-0.571429
49859,1917-10-07,Uruguay,0.0,4.0,-4.0,-1.200000,-1.000000
521,1917-10-12,Chile,5.0,0.0,5.0,-0.200000,-0.333333
49863,1917-10-16,Uruguay,1.0,3.0,-2.0,-0.400000,-0.500000


In [67]:
goal_diff_features = team_form[['date', 'team', 'goal_diff_5', 'goal_diff_10']].copy()
historical_results = pd.merge(historical_results, goal_diff_features, left_on=['date', 'home_team'], right_on=['date', 'team'])
historical_results = historical_results.rename(columns={
    'goal_diff_5': 'home_goal_diff_5',
    'goal_diff_10': 'home_goal_diff_10'
})

historical_results = pd.merge(historical_results, goal_diff_features, left_on=['date', 'away_team'], right_on=['date', 'team'])
historical_results = historical_results.rename(columns={
    'goal_diff_5': 'away_goal_diff_5',
    'goal_diff_10': 'away_goal_diff_10'
})


In [68]:
print(historical_results.shape)
print(historical_results.columns.tolist())

(27126, 31)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_rank', 'home_points', 'away_rank', 'away_points', 'points_diff', 'home_form_5', 'home_form_10', 'away_form_5', 'away_form_10', 'team1', 'team2', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw', 'team_x', 'home_goal_diff_5', 'home_goal_diff_10', 'team_y', 'away_goal_diff_5', 'away_goal_diff_10']


In [69]:
historical_results = historical_results.drop(columns=['team_x', 'team_y'])
historical_results = historical_results.drop_duplicates(subset=['date', 'home_team', 'away_team'], keep='first')

In [70]:
print(historical_results.shape)
print(historical_results.columns.tolist())

(27110, 29)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_rank', 'home_points', 'away_rank', 'away_points', 'points_diff', 'home_form_5', 'home_form_10', 'away_form_5', 'away_form_10', 'team1', 'team2', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw', 'home_goal_diff_5', 'home_goal_diff_10', 'away_goal_diff_5', 'away_goal_diff_10']


In [71]:
historical_results.to_csv('../data/processed/historical_results_features.csv', index=False)

In [5]:
player_valuations_clean = pd.merge(player_valuations_clean, players_clean, on='player_id')

In [6]:
print(player_valuations_clean.shape)
print(player_valuations_clean.columns.tolist())

(503962, 10)
['player_id', 'date', 'market_value_in_eur_x', 'current_club_name', 'current_club_id', 'name', 'country_of_citizenship', 'position', 'international_caps', 'market_value_in_eur_y']


In [7]:
player_valuations_clean = player_valuations_clean[['player_id', 'date', 'market_value_in_eur_x', 'country_of_citizenship']].copy()

player_valuations_clean = player_valuations_clean.rename(columns={'market_value_in_eur_x': 'market_value_in_eur'})

In [ ]:
squad_values = (player_valuations_clean.groupby(['country_of_citizenship', 'date'])['market_value_in_eur'].sum().reset_index())

In [9]:
print(squad_values.shape)
squad_values.head(10)

(89221, 3)


,country_of_citizenship,date,market_value_in_eur
0,Afghanistan,2011-11-14,25000
1,Afghanistan,2012-04-24,100000
2,Afghanistan,2012-07-02,350000
3,Afghanistan,2013-01-13,350000
4,Afghanistan,2013-06-19,300000
5,Afghanistan,2013-08-07,25000
6,Afghanistan,2014-02-10,100000
7,Afghanistan,2014-03-28,300000
8,Afghanistan,2014-08-15,150000
9,Afghanistan,2015-02-21,150000


In [13]:
home_squad_values = squad_values.copy()
home_squad_values = home_squad_values.rename(columns={
    'country_of_citizenship': 'home_team',
    'market_value_in_eur': 'home_squad_value'
})

away_squad_values = squad_values.copy()
away_squad_values = away_squad_values.rename(columns={
    'country_of_citizenship': 'away_team',
    'market_value_in_eur': 'away_squad_value'
})

home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')
historical_results = historical_results.sort_values(by='date')

historical_results = pd.merge_asof(historical_results, home_squad_values, left_on = 'date', right_on = 'date', by = 'home_team', direction = 'backward')
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on = 'date', right_on = 'date', by = 'away_team', direction = 'backward')

MergeError: Incompatible merge dtype, dtype('<M8[us]') and <StringDtype(storage='python', na_value=nan)>, both sides must have numeric dtype

In [11]:
print(historical_results['date'].dtype)
print(home_squad_values['date'].dtype)

str
str


In [14]:
historical_results['date'] = pd.to_datetime(historical_results['date'])
home_squad_values['date'] = pd.to_datetime(home_squad_values['date'])
away_squad_values['date'] = pd.to_datetime(away_squad_values['date'])

In [17]:
home_squad_values = squad_values.copy()
home_squad_values = home_squad_values.rename(columns={
    'country_of_citizenship': 'home_team',
    'market_value_in_eur': 'home_squad_value'
})

away_squad_values = squad_values.copy()
away_squad_values = away_squad_values.rename(columns={
    'country_of_citizenship': 'away_team',
    'market_value_in_eur': 'away_squad_value'
})

home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')
historical_results = historical_results.sort_values(by='date')

historical_results = pd.merge_asof(historical_results, home_squad_values, left_on = 'date', right_on = 'date', by = 'home_team', direction = 'backward')
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on = 'date', right_on = 'date', by = 'away_team', direction = 'backward')

MergeError: Incompatible merge dtype, dtype('<M8[us]') and <StringDtype(storage='python', na_value=nan)>, both sides must have numeric dtype

In [18]:
player_valuations_clean['date'] = pd.to_datetime(player_valuations_clean['date'])

In [19]:
home_squad_values = squad_values.copy()
home_squad_values = home_squad_values.rename(columns={
    'country_of_citizenship': 'home_team',
    'market_value_in_eur': 'home_squad_value'
})

away_squad_values = squad_values.copy()
away_squad_values = away_squad_values.rename(columns={
    'country_of_citizenship': 'away_team',
    'market_value_in_eur': 'away_squad_value'
})

home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')
historical_results = historical_results.sort_values(by='date')

historical_results = pd.merge_asof(historical_results, home_squad_values, left_on = 'date', right_on = 'date', by = 'home_team', direction = 'backward')
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on = 'date', right_on = 'date', by = 'away_team', direction = 'backward')

MergeError: Incompatible merge dtype, dtype('<M8[us]') and <StringDtype(storage='python', na_value=nan)>, both sides must have numeric dtype

In [20]:
print(historical_results['date'].dtype)
print(home_squad_values['date'].dtype)
print(away_squad_values['date'].dtype)

datetime64[us]
str
str


In [21]:
home_squad_values['date'] = pd.to_datetime(home_squad_values['date'])
away_squad_values['date'] = pd.to_datetime(away_squad_values['date'])

In [23]:
home_squad_values = squad_values.copy()
home_squad_values = home_squad_values.rename(columns={
    'country_of_citizenship': 'home_team',
    'market_value_in_eur': 'home_squad_value'
})

away_squad_values = squad_values.copy()
away_squad_values = away_squad_values.rename(columns={
    'country_of_citizenship': 'away_team',
    'market_value_in_eur': 'away_squad_value'
})

home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')
historical_results = historical_results.sort_values(by='date')

home_squad_values['date'] = pd.to_datetime(home_squad_values['date'])
away_squad_values['date'] = pd.to_datetime(away_squad_values['date'])

historical_results = pd.merge_asof(historical_results, home_squad_values, left_on = 'date', right_on = 'date', by = 'home_team', direction = 'backward')
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on = 'date', right_on = 'date', by = 'away_team', direction = 'backward')

In [24]:
print(historical_results.shape)
historical_results[['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value']].head(10)

(49339, 13)


,date,home_team,away_team,home_squad_value,away_squad_value
0,1872-11-30,Scotland,England,NaN,NaN
1,1873-03-08,England,Scotland,NaN,NaN
2,1874-03-07,Scotland,England,NaN,NaN
3,1875-03-06,England,Scotland,NaN,NaN
4,1876-03-04,Scotland,England,NaN,NaN
5,1876-03-25,Scotland,Wales,NaN,NaN
6,1877-03-03,England,Scotland,NaN,NaN
7,1877-03-05,Wales,Scotland,NaN,NaN
8,1878-03-02,Scotland,England,NaN,NaN
9,1878-03-23,Scotland,Wales,NaN,NaN


In [27]:
historical_results[historical_results['date'] > '2000-01-20'][['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value']].head(10)

,date,home_team,away_team,home_squad_value,away_squad_value
24081,2000-01-21,New Zealand,Korea Republic,NaN,NaN
24082,2000-01-22,Ghana,Cameroon,NaN,NaN
24083,2000-01-23,China PR,Philippines,NaN,NaN
24084,2000-01-23,Egypt,Zambia,NaN,NaN
24085,2000-01-23,New Zealand,Korea Republic,NaN,NaN
24086,2000-01-23,Nigeria,Tunisia,NaN,NaN
24087,2000-01-23,South Africa,Gabon,NaN,NaN
24088,2000-01-23,Vietnam,Guam,NaN,NaN
24089,2000-01-24,Côte d'Ivoire,Togo,NaN,150000.0
24090,2000-01-24,Qatar,Bosnia and Herzegovina,NaN,NaN


In [26]:
print(squad_values['date'].min())

2000-01-20


In [28]:
historical_results[historical_results['date'] > '2000-02-01'][['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value']].head(10)

,date,home_team,away_team,home_squad_value,away_squad_value
24121,2000-02-02,Romania,Latvia,NaN,NaN
24122,2000-02-02,Saint Kitts and Nevis,Saint Lucia,NaN,NaN
24123,2000-02-02,South Africa,Algeria,NaN,NaN
24124,2000-02-02,Finland,Iceland,NaN,NaN
24125,2000-02-02,Cyprus,Lithuania,NaN,NaN
24126,2000-02-02,Costa Rica,Chile,NaN,NaN
24127,2000-02-02,Congo DR,Gabon,NaN,NaN
24128,2000-02-02,Armenia,Moldova,NaN,NaN
24129,2000-02-03,Nigeria,Morocco,NaN,NaN
24130,2000-02-03,Tunisia,Congo,NaN,NaN


In [29]:
print(squad_values['country_of_citizenship'].unique()[:30])

<StringArray>
[        'Afghanistan',             'Albania',             'Algeria',
             'Andorra',              'Angola', 'Antigua and Barbuda',
           'Argentina',             'Armenia',               'Aruba',
           'Australia',             'Austria',          'Azerbaijan',
             'Bahrain',          'Bangladesh',            'Barbados',
             'Belarus',             'Belgium',               'Benin',
             'Bermuda',             'Bolivia',             'Bonaire',
  'Bosnia-Herzegovina',              'Brazil',   'Brunei Darussalam',
            'Bulgaria',        'Burkina Faso',             'Burundi',
            'Cameroon',              'Canada',          'Cape Verde']
Length: 30, dtype: str


In [30]:
# Check what Bosnia is called in historical_results
print(historical_results[historical_results['home_team'].str.contains('Bosnia', na=False)]['home_team'].unique())

# Check what Côte d'Ivoire is called in squad_values
print(squad_values[squad_values['country_of_citizenship'].str.contains('Ivoire', na=False)]['country_of_citizenship'].unique())

# Check Korea
print(squad_values[squad_values['country_of_citizenship'].str.contains('Korea', na=False)]['country_of_citizenship'].unique())

# Check USA
print(squad_values[squad_values['country_of_citizenship'].str.contains('States', na=False)]['country_of_citizenship'].unique())

<StringArray>
['Bosnia and Herzegovina']
Length: 1, dtype: str
<StringArray>
['Cote d'Ivoire']
Length: 1, dtype: str
<StringArray>
['Korea, North', 'Korea, South']
Length: 2, dtype: str
<StringArray>
['United States']
Length: 1, dtype: str


In [31]:
print(squad_values[squad_values['country_of_citizenship'].str.contains('Iran', na=False)]['country_of_citizenship'].unique())
print(squad_values[squad_values['country_of_citizenship'].str.contains('Congo', na=False)]['country_of_citizenship'].unique())
print(squad_values[squad_values['country_of_citizenship'].str.contains('Turkey|Turk', na=False)]['country_of_citizenship'].unique())
print(squad_values[squad_values['country_of_citizenship'].str.contains('Czech', na=False)]['country_of_citizenship'].unique())

<StringArray>
['Iran']
Length: 1, dtype: str
<StringArray>
['Congo', 'DR Congo']
Length: 2, dtype: str
<StringArray>
['Turkey', 'Turkmenistan']
Length: 2, dtype: str
<StringArray>
['Czech Republic']
Length: 1, dtype: str


In [35]:
squad_values = squad_values.replace({
    'Bosnia-Herzegovina' : 'Bosnia and Herzegovina',
    "Cote d'Ivoire" : "Côte d'Ivoire",
    'Korea, South' : 'Korea Republic',
    'Korea, North' : 'Korea DPR',
    'United States' : 'USA',
    'DR Congo' : 'Congo DR',
    'Turkey' : 'Türkiye',
    'Czech Republic' : 'Czechia',
    'Iran' : 'IR Iran'
})

In [36]:
home_squad_values = squad_values.copy()
home_squad_values = home_squad_values.rename(columns={
    'country_of_citizenship': 'home_team',
    'market_value_in_eur': 'home_squad_value'
})

away_squad_values = squad_values.copy()
away_squad_values = away_squad_values.rename(columns={
    'country_of_citizenship': 'away_team',
    'market_value_in_eur': 'away_squad_value'
})

home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')
historical_results = historical_results.sort_values(by='date')

home_squad_values['date'] = pd.to_datetime(home_squad_values['date'])
away_squad_values['date'] = pd.to_datetime(away_squad_values['date'])

historical_results = pd.merge_asof(historical_results, home_squad_values, left_on = 'date', right_on = 'date', by = 'home_team', direction = 'backward')
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on = 'date', right_on = 'date', by = 'away_team', direction = 'backward')

In [38]:
print(historical_results.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_squad_value_x', 'away_squad_value_x', 'home_squad_value_y', 'away_squad_value_y']


In [39]:
historical_results = historical_results.drop(columns=['home_squad_value_x', 'away_squad_value_x'])
historical_results = historical_results.rename(columns={
    'home_squad_value_y': 'home_squad_value',
    'away_squad_value_y': 'away_squad_value'
})

In [40]:
historical_results[historical_results['date'] > '2000-06-01'][['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value']].head(10)

,date,home_team,away_team,home_squad_value,away_squad_value
24553,2000-06-02,Lesotho,Mozambique,NaN,NaN
24554,2000-06-02,IR Iran,Syria,NaN,NaN
24555,2000-06-02,Jordan,Iraq,NaN,NaN
24556,2000-06-02,Singapore,Qatar,NaN,NaN
24557,2000-06-02,Eswatini,Botswana,NaN,NaN
24558,2000-06-02,Portugal,Wales,NaN,NaN
24559,2000-06-03,Algeria,Guinea,NaN,NaN
24560,2000-06-03,Germany,Czechia,NaN,NaN
24561,2000-06-03,Hungary,Israel,NaN,NaN
24562,2000-06-03,Honduras,Haiti,NaN,NaN


In [41]:
print(historical_results.shape)
print(historical_results.columns.tolist())

(49339, 13)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_squad_value', 'away_squad_value']


In [42]:
historical_results = pd.read_csv('../data/processed/historical_results_features.csv')
historical_results['date'] = pd.to_datetime(historical_results['date'])

ValueError: time data "--" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [43]:
historical_results = pd.read_csv('../data/processed/historical_results_features.csv')
historical_results['date'] = pd.to_datetime(historical_results['date'], errors='coerce')

In [44]:
print(historical_results.shape)
print(historical_results.columns.tolist())

(27111, 29)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_rank', 'home_points', 'away_rank', 'away_points', 'points_diff', 'home_form_5', 'home_form_10', 'away_form_5', 'away_form_10', 'team1', 'team2', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw', 'home_goal_diff_5', 'home_goal_diff_10', 'away_goal_diff_5', 'away_goal_diff_10']


In [47]:
historical_results = historical_results.dropna(subset=['date'])

In [49]:
# Sort all three
historical_results = historical_results.sort_values(by='date')
home_squad_values = home_squad_values.sort_values(by='date')
away_squad_values = away_squad_values.sort_values(by='date')

# Merge home squad value
historical_results = pd.merge_asof(historical_results, home_squad_values, left_on='date', right_on='date', by='home_team', direction='backward')

# Merge away squad value
historical_results = pd.merge_asof(historical_results, away_squad_values, left_on='date', right_on='date', by='away_team', direction='backward')

In [50]:
historical_results[historical_results['date'] > '2000-06-01'][['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value']].head(10)

,date,home_team,away_team,home_squad_value,away_squad_value
5064,2000-06-02,IR Iran,Syria,NaN,NaN
5065,2000-06-02,Jordan,Iraq,NaN,NaN
5066,2000-06-02,Lesotho,Mozambique,NaN,NaN
5067,2000-06-02,Portugal,Wales,NaN,NaN
5068,2000-06-02,Singapore,Qatar,NaN,NaN
5069,2000-06-02,Eswatini,Botswana,NaN,NaN
5070,2000-06-03,Algeria,Guinea,NaN,NaN
5071,2000-06-03,Denmark,Belgium,NaN,NaN
5072,2000-06-03,Honduras,Haiti,NaN,NaN
5073,2000-06-03,Germany,Czechia,NaN,NaN


In [51]:
# Check what Portugal looks like in squad_values
print(squad_values[squad_values['country_of_citizenship'] == 'Portugal']['date'].head())

# Check what the earliest non-null squad value date is
print(squad_values['date'].min())
print(squad_values['date'].max())

63869    2004-10-04
63870    2004-10-09
63871    2004-10-12
63872    2004-11-22
63873    2004-11-23
Name: date, dtype: str
2000-01-20
2026-02-27


## Squad Value Features — Final Merge

Rebuild squad values with complete country name mapping, merge onto the 29-column feature CSV, impute pre-2004 NaNs with global median, and derive relative features.

In [52]:
SQUAD_VALUE_NAME_MAP = {
    'Bosnia-Herzegovina':   'Bosnia and Herzegovina',
    "Cote d'Ivoire":        "Côte d'Ivoire",
    'Korea, South':         'Korea Republic',
    'Korea, North':         'Korea DPR',
    'United States':        'USA',
    'DR Congo':             'Congo DR',
    'Turkey':               'Türkiye',
    'Czech Republic':       'Czechia',
    'Iran':                 'IR Iran',
    'Cape Verde':           'Cabo Verde',
    'Ireland':              'Republic of Ireland',
    'China':                'China PR',
    'Curacao':              'Curaçao',
}

# Rebuild squad value timeseries with complete name map
# player_valuations_clean already has country_of_citizenship merged in from earlier cells
squad_values_mapped = (
    player_valuations_clean
    .groupby(['country_of_citizenship', 'date'])['market_value_in_eur']
    .sum()
    .reset_index()
    .rename(columns={'market_value_in_eur': 'squad_value_eur'})
)
squad_values_mapped['country_of_citizenship'] = squad_values_mapped['country_of_citizenship'].replace(SQUAD_VALUE_NAME_MAP)
squad_values_mapped['date'] = pd.to_datetime(squad_values_mapped['date'])
squad_values_mapped = squad_values_mapped.sort_values('date')

# Build home and away versions for merge_asof
home_sv = squad_values_mapped.rename(columns={
    'country_of_citizenship': 'home_team',
    'squad_value_eur': 'home_squad_value'
})
away_sv = squad_values_mapped.rename(columns={
    'country_of_citizenship': 'away_team',
    'squad_value_eur': 'away_squad_value'
})

# Load clean 29-column base (drop the '--' garbage date row)
historical_results = pd.read_csv('../data/processed/historical_results_features.csv')
historical_results['date'] = pd.to_datetime(historical_results['date'], errors='coerce')
historical_results = historical_results.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

# Merge squad values using most-recent snapshot before each match date
historical_results = pd.merge_asof(historical_results, home_sv, on='date', by='home_team', direction='backward')
historical_results = pd.merge_asof(historical_results, away_sv, on='date', by='away_team', direction='backward')

print(historical_results.shape)
print(historical_results[['home_squad_value', 'away_squad_value']].isnull().sum())

(27110, 31)
home_squad_value    11676
away_squad_value    11835
dtype: int64


In [53]:
# Impute pre-2004 NaNs with global median (not 0 — 0 would create false signal)
global_median_sv = squad_values_mapped['squad_value_eur'].median()
print(f"Global median squad value: €{global_median_sv:,.0f}")

historical_results['home_squad_value'] = historical_results['home_squad_value'].fillna(global_median_sv)
historical_results['away_squad_value'] = historical_results['away_squad_value'].fillna(global_median_sv)

# Derived relative squad strength features
historical_results['squad_value_diff'] = historical_results['home_squad_value'] - historical_results['away_squad_value']
historical_results['squad_value_ratio'] = historical_results['home_squad_value'] / (historical_results['away_squad_value'] + 1)

print(f"NaN remaining — home: {historical_results['home_squad_value'].isnull().sum()}, away: {historical_results['away_squad_value'].isnull().sum()}")
print(f"Final shape: {historical_results.shape}")
historical_results[['date', 'home_team', 'away_team', 'home_squad_value', 'away_squad_value', 'squad_value_diff']].tail(5)

Global median squad value: €1,475,000
NaN remaining — home: 0, away: 0
Final shape: (27110, 33)


,date,home_team,away_team,home_squad_value,away_squad_value,squad_value_diff
27105,2026-06-05,Azerbaijan,Malta,350000.0,2500000.0,-2150000.0
27106,2026-06-05,Angola,Mauritania,225000.0,1000000.0,-775000.0
27107,2026-06-05,Puerto Rico,Saudi Arabia,1475000.0,50000.0,1425000.0
27108,2026-06-05,San Marino,Bangladesh,50000.0,5000000.0,-4950000.0
27109,2026-06-05,Vanuatu,Fiji,1475000.0,1475000.0,0.0


In [54]:
historical_results.to_csv('../data/processed/historical_results_features.csv', index=False)
print(f"Saved {historical_results.shape[0]} rows × {historical_results.shape[1]} columns")
print(historical_results.columns.tolist())

Saved 27110 rows × 33 columns
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome', 'tournament_weight', 'home_rank', 'home_points', 'away_rank', 'away_points', 'points_diff', 'home_form_5', 'home_form_10', 'away_form_5', 'away_form_10', 'team1', 'team2', 'h2h_team1_wins', 'h2h_team2_wins', 'h2h_draw', 'home_goal_diff_5', 'home_goal_diff_10', 'away_goal_diff_5', 'away_goal_diff_10', 'home_squad_value', 'away_squad_value', 'squad_value_diff', 'squad_value_ratio']
